# Modelo Supervisado - Sistema Experto para Diagnóstico Básico de Fallas en Redes
**Taller No. 2 - Sistemas Expertos e Inteligencia Artificial**

Este notebook implementa un modelo de **clasificación supervisada** (Árbol de Decisión y Random Forest) que complementa el sistema experto basado en reglas, aprendiendo a predecir el `diagnostico` de una falla de red a partir de los síntomas reportados (variables categóricas como tipo de conexión, estado del LED del router, resultado de ping, etc.).

In [ ]:
# 1. Instalación e importación de librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

## 2. Carga del dataset

El dataset (`dataset_diagnostico_red.csv`) fue construido de forma sintética a partir de las mismas reglas del sistema experto (punto 6 del taller), simulando 600 casos de soporte técnico con 8 variables de entrada (síntomas) y 1 variable objetivo (`diagnostico`).

> En Google Colab: sube el archivo `dataset_diagnostico_red.csv` con `files.upload()` o móntalo desde Google Drive.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # seleccionar dataset_diagnostico_red.csv

df = pd.read_csv('dataset_diagnostico_red.csv')
print(df.shape)
df.head()

In [ ]:
# 3. Análisis exploratorio rápido
df['diagnostico'].value_counts().plot(kind='bar', figsize=(9,4), color='#4472C4')
plt.title('Distribución de clases de diagnóstico')
plt.ylabel('Número de casos')
plt.tight_layout()
plt.show()

## 4. Preprocesamiento

Todas las variables son categóricas, por lo que se codifican con `OrdinalEncoder` (variables de entrada) y `LabelEncoder` (variable objetivo). Se separa el 80% para entrenamiento y 20% para prueba, de forma estratificada para conservar la proporción de clases.

In [ ]:
X = df.drop(columns=['diagnostico'])
y = df['diagnostico']

encoder_X = OrdinalEncoder()
X_enc = encoder_X.fit_transform(X)

encoder_y = LabelEncoder()
y_enc = encoder_y.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print('Entrenamiento:', X_train.shape, ' Prueba:', X_test.shape)

## 5. Entrenamiento de modelos

Se entrenan dos modelos de clasificación:
1. **Árbol de Decisión** — fácil de interpretar y visualizar (similar en espíritu a las reglas IF-THEN del sistema experto).
2. **Random Forest** — conjunto de árboles que generalmente mejora la exactitud y reduce el sobreajuste.

In [ ]:
arbol = DecisionTreeClassifier(max_depth=5, random_state=42)
arbol.fit(X_train, y_train)

bosque = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
bosque.fit(X_train, y_train)

print('Modelos entrenados correctamente')

## 6. Evaluación con métricas: Accuracy, Precision, Recall y F1-Score

In [ ]:
def evaluar(modelo, nombre):
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    print(f'--- {nombre} ---')
    print(f'Accuracy : {acc:.3f}')
    print(f'Precision (macro): {prec:.3f}')
    print(f'Recall (macro)   : {rec:.3f}')
    print(f'F1-Score (macro) : {f1:.3f}')
    print()
    return y_pred

y_pred_arbol = evaluar(arbol, 'Árbol de Decisión')
y_pred_bosque = evaluar(bosque, 'Random Forest')

In [ ]:
print(classification_report(y_test, y_pred_bosque, target_names=encoder_y.classes_, zero_division=0))

cm = confusion_matrix(y_test, y_pred_bosque)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=encoder_y.classes_)
fig, ax = plt.subplots(figsize=(8,8))
disp.plot(ax=ax, xticks_rotation=90, colorbar=False, cmap='Blues')
plt.title('Matriz de confusión - Random Forest')
plt.tight_layout()
plt.show()

## 7. Importancia de variables (interpretabilidad, análogo al módulo de explicación del sistema experto)

In [ ]:
importancias = pd.Series(bosque.feature_importances_, index=X.columns).sort_values(ascending=False)
importancias.plot(kind='barh', figsize=(7,4), color='#70AD47')
plt.title('Importancia de variables - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
importancias

## 8. Conclusión

El modelo de **Random Forest** obtiene mejor desempeño general que el árbol individual y permite además cuantificar qué síntomas (variables) son más determinantes para cada diagnóstico, lo cual es coherente con las reglas priorizadas por el sistema experto (por ejemplo, `led_enlace_router` y `ping_router` resultan altamente informativas, tal como en las Reglas 1, 2 y 4 del punto 6 del taller). Este modelo puede integrarse al sistema experto como un **segundo motor de diagnóstico probabilístico**, útil quando los síntomas no calzan exactamente en ninguna regla IF-THEN predefinida.